In [3]:
from pathlib import Path
import re
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np


# =========================
# User settings
# =========================
input_folder = r"raw_logos"
output_folder = r"public/Figure/University"

output_width = 256
output_height = 256


recursive = False

# Preserve near-white pixels as white.
# 255 = only pure white
# 245 = preserve near-white / anti-aliased white background
white_threshold = 245

# Raster outputs are saved as PNG because PNG supports transparency.
raster_output_ext = ".png"


# =========================
# SVG functions
# =========================
SVG_NS = "http://www.w3.org/2000/svg"
ET.register_namespace("", SVG_NS)

COLOR_ATTRS = {
    "fill",
    "stroke",
    "color",
    "stop-color",
    "flood-color",
    "lighting-color",
}


def strip_namespace(tag):
    if "}" in tag:
        return tag.split("}", 1)[1]
    return tag


def is_transparent_or_empty(value):
    if value is None:
        return True

    v = value.strip().lower()

    if v in {"", "none", "transparent"}:
        return True

    if v.startswith("rgba"):
        nums = re.findall(r"[-+]?\d*\.?\d+", v)
        if nums and float(nums[-1]) == 0:
            return True

    if v.startswith("hsla"):
        nums = re.findall(r"[-+]?\d*\.?\d+", v)
        if nums and float(nums[-1]) == 0:
            return True

    return False


def is_white_svg_color(value):
    if value is None:
        return False

    v = value.strip().lower().replace(" ", "")

    if v in {"white", "#fff", "#ffffff", "rgb(255,255,255)", "rgba(255,255,255,1)"}:
        return True

    # rgb(255, 255, 255) or rgba(255, 255, 255, alpha)
    if v.startswith("rgb"):
        nums = re.findall(r"[-+]?\d*\.?\d+", v)
        if len(nums) >= 3:
            r, g, b = float(nums[0]), float(nums[1]), float(nums[2])
            return r >= white_threshold and g >= white_threshold and b >= white_threshold

    return False


def make_black_except_white(value):
    if is_transparent_or_empty(value):
        return value

    if is_white_svg_color(value):
        return value

    # Do not destroy references such as url(#gradient)
    if value.strip().lower().startswith("url("):
        return value

    return "#000000"


def recolor_style_attribute(style):
    if not style:
        return style

    parts = style.split(";")
    new_parts = []

    for part in parts:
        if ":" not in part:
            new_parts.append(part)
            continue

        key, value = part.split(":", 1)
        key_clean = key.strip().lower()

        if key_clean in COLOR_ATTRS:
            value = make_black_except_white(value)

        new_parts.append(f"{key}:{value}")

    return ";".join(new_parts)


def recolor_css_text(css):
    def replace_property(match):
        prop = match.group(1)
        value = match.group(2).strip()

        new_value = make_black_except_white(value)
        return f"{prop}: {new_value}"

    pattern = r"\b(fill|stroke|color|stop-color|flood-color|lighting-color)\s*:\s*([^;}\n]+)"
    return re.sub(pattern, replace_property, css, flags=re.IGNORECASE)


def parse_size(value):
    if value is None:
        return None

    match = re.search(r"[-+]?\d*\.?\d+", value)
    if match:
        return float(match.group())

    return None


def ensure_viewbox(root):
    if "viewBox" in root.attrib:
        return

    width = parse_size(root.attrib.get("width"))
    height = parse_size(root.attrib.get("height"))

    if width is not None and height is not None:
        root.set("viewBox", f"0 0 {width:g} {height:g}")


def resize_svg(root, width, height):
    ensure_viewbox(root)

    root.set("width", str(width))
    root.set("height", str(height))

    if "preserveAspectRatio" not in root.attrib:
        root.set("preserveAspectRatio", "xMidYMid meet")


def recolor_svg(root):
    for elem in root.iter():
        tag = strip_namespace(elem.tag)

        if tag == "style" and elem.text:
            elem.text = recolor_css_text(elem.text)

        for attr_name in list(elem.attrib.keys()):
            clean_attr = attr_name.split("}", 1)[-1].lower()

            if clean_attr in COLOR_ATTRS:
                elem.set(attr_name, make_black_except_white(elem.attrib[attr_name]))

            elif clean_attr == "style":
                elem.set(attr_name, recolor_style_attribute(elem.attrib[attr_name]))


def process_svg(input_path, output_path, width, height):
    parser = ET.XMLParser(target=ET.TreeBuilder(insert_comments=True))
    tree = ET.parse(input_path, parser=parser)
    root = tree.getroot()

    resize_svg(root, width, height)
    recolor_svg(root)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)


# =========================
# PNG / JPG functions
# =========================
def resize_with_padding_rgba(img, target_width, target_height):
    """
    Resize while preserving aspect ratio.
    Transparent padding is added if aspect ratio does not match.
    """
    img = img.convert("RGBA")

    original_width, original_height = img.size

    scale = min(
        target_width / original_width,
        target_height / original_height,
    )

    new_width = int(original_width * scale)
    new_height = int(original_height * scale)

    resized = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

    canvas = Image.new("RGBA", (target_width, target_height), (0, 0, 0, 0))

    x_offset = (target_width - new_width) // 2
    y_offset = (target_height - new_height) // 2

    canvas.paste(resized, (x_offset, y_offset), resized)

    return canvas


def raster_to_black_keep_white(
    input_path,
    output_path,
    width,
    height,
    white_threshold=245,
):
    img = Image.open(input_path).convert("RGBA")
    arr = np.array(img)

    rgb = arr[:, :, :3]
    alpha = arr[:, :, 3]

    visible_mask = alpha > 0

    white_mask = (
        (rgb[:, :, 0] >= white_threshold) &
        (rgb[:, :, 1] >= white_threshold) &
        (rgb[:, :, 2] >= white_threshold)
    )

    # Change only visible non-white pixels to black
    black_mask = visible_mask & (~white_mask)

    arr[:, :, 0][black_mask] = 0
    arr[:, :, 1][black_mask] = 0
    arr[:, :, 2][black_mask] = 0

    # Keep white pixels white
    arr[:, :, 0][visible_mask & white_mask] = 255
    arr[:, :, 1][visible_mask & white_mask] = 255
    arr[:, :, 2][visible_mask & white_mask] = 255

    result = Image.fromarray(arr, mode="RGBA")

    result = resize_with_padding_rgba(
        result,
        target_width=width,
        target_height=height,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.save(output_path)


# =========================
# Batch processing
# =========================
input_dir = Path(input_folder)
output_dir = Path(output_folder)

output_dir.mkdir(parents=True, exist_ok=True)

valid_exts = {".svg", ".png", ".jpg", ".jpeg"}

if recursive:
    files = sorted([p for p in input_dir.rglob("*") if p.suffix.lower() in valid_exts])
else:
    files = sorted([p for p in input_dir.glob("*") if p.suffix.lower() in valid_exts])

print(f"Found {len(files)} files.")

for input_path in files:
    ext = input_path.suffix.lower()

    if recursive:
        relative_path = input_path.relative_to(input_dir)
    else:
        relative_path = Path(input_path.name)

    if ext == ".svg":
        output_path = output_dir / relative_path
        process_svg(
            input_path=input_path,
            output_path=output_path,
            width=output_width,
            height=output_height,
        )

    elif ext in {".png", ".jpg", ".jpeg"}:
        output_path = output_dir / relative_path.with_suffix(raster_output_ext)

        raster_to_black_keep_white(
            input_path=input_path,
            output_path=output_path,
            width=output_width,
            height=output_height,
            white_threshold=white_threshold,
        )

    print(f"Saved: {output_path}")

print("Done.")

Found 11 files.
Saved: public\Figure\University\AIT.png
Saved: public\Figure\University\frankfurt.png
Saved: public\Figure\University\imperial.png
Saved: public\Figure\University\LBS.png
Saved: public\Figure\University\LMU.png
Saved: public\Figure\University\maxplanck.png
Saved: public\Figure\University\oxford.png
Saved: public\Figure\University\RWTH.png
Saved: public\Figure\University\syney.png
Saved: public\Figure\University\TU Delft.png
Saved: public\Figure\University\TUMunich.png
Done.
